# Tutorial 00: Shared Reusable Agents for A2A Foundations

A production-grade walkthrough of the reusable agent services in `A2A/00_Agents`, designed as the base layer for all later A2A tutorials.

## 1) Where We Are in the Journey

This is the foundation step of the A2A track, so there is no previous numbered tutorial to rely on yet.
In this step, we establish standalone, reusable agent services with clear discovery and invocation contracts.
Every later controller, router, or orchestrator depends on these service interfaces being stable and predictable.

## 2) What You Will Learn

By the end of this tutorial, you will understand:

- How reusable agent services expose capabilities (`/tools`) and identity (`/agent-info`)
- How invocation payloads map to tool execution (`/invoke`)
- Why interface consistency matters more than internal implementation details in multi-agent systems
- How to reason about synchronous vs delayed/async behavior as part of system design
- How this folder becomes shared infrastructure for all subsequent A2A lessons

## 3) Why This Matters

Real A2A systems fail less from model quality and more from contract mismatch between agents.
This folder encodes the minimal service contract needed for discovery, routing, and invocation patterns that align with MCP-style tool exposure.
If these interfaces are clean, higher-level orchestrators can remain generic, reusable, and easier to test.

## 4) Core Concept Deep Dive

### Separation of concerns in agent services

This folder demonstrates a crucial split:

- **Discovery plane**: `GET /tools` exposes capability schemas (what can be done)
- **Identity plane**: `GET /agent-info` exposes who the agent is and where it can be invoked
- **Execution plane**: `POST /invoke` executes the request and returns result/error

This three-plane split is a strong design choice because orchestrators can discover and reason about capabilities before executing anything.

### Intuition

Think of each agent as a microservice with self-describing tools. The controller does not need hardcoded behavior for each agent; it can inspect available tools and route requests dynamically.

### Trade-offs

- **Pros**: composability, easier orchestration, easier testing, cleaner contracts
- **Cons**: extra endpoint surface area, need for strict schema discipline, versioning concerns

### System design takeaway

The service API shape in this folder is more important than the arithmetic/finance logic itself. These files teach interface-first agent engineering.

## 5) Code Walkthrough (Only `A2A/00_Agents`)

Primary files and roles:

- `math_agent.py`: baseline math agent (`add`)
- `finance_agent.py`: baseline finance agent (`calculate_interest`)
- `math_agent_delay.py`: sync delay simulation via `time.sleep(20)`
- `finance_agent_delay.py`: sync delay simulation for finance flow
- `math_agent_delay-async.py`: async variant (contains a syntax issue worth noting)
- `finance_agent_delay-async.py`: async delayed finance invocation

Common endpoint contract in each service:

1. `GET /tools` returns structured tool metadata and input schema.
2. `GET /agent-info` returns name, description, and endpoint location.
3. `POST /invoke` receives payload and returns `{status, result}` or `{status, message}`.

This consistency is what lets later folders build routing logic without rewriting per-agent adapters.

In [1]:
# 6) Executable Cell: Setup and file inventory
from pathlib import Path

BASE = Path('.').resolve()
agent_files = [
    'math_agent.py',
    'finance_agent.py',
    'math_agent_delay.py',
    'finance_agent_delay.py',
    'math_agent_delay-async.py',
    'finance_agent_delay-async.py',
]

print('Notebook working directory:', BASE)
for f in agent_files:
    p = BASE / f
    print(f'{f:30} exists={p.exists()}')

Notebook working directory: /home/navinkumar_25_gmail_com/A2A/00_Agents
math_agent.py                  exists=True
finance_agent.py               exists=True
math_agent_delay.py            exists=True
finance_agent_delay.py         exists=True
math_agent_delay-async.py      exists=True
finance_agent_delay-async.py   exists=True


In [2]:
# 7) Executable Cell: Import baseline agents and inspect discovery endpoints
import importlib.util

def load_module(module_name: str, file_name: str):
    file_path = BASE / file_name
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

math_mod = load_module('math_agent_mod', 'math_agent.py')
finance_mod = load_module('finance_agent_mod', 'finance_agent.py')

print('Math tools:', math_mod.get_tools())
print('Math agent-info:', math_mod.agent_info())
print('Finance tools:', finance_mod.get_tools())
print('Finance agent-info:', finance_mod.agent_info())

Math tools: {'tools': [{'name': 'add', 'description': 'Add two numbers', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}}]}
Math agent-info: {'name': 'math-agent', 'description': 'Handles mathematical operations like addition', 'endpoint': 'http://localhost:8000/invoke'}
Finance tools: {'tools': [{'name': 'calculate_interest', 'description': 'Calculate simple interest', 'input_schema': {'type': 'object', 'properties': {'principal': {'type': 'number'}, 'rate': {'type': 'number'}, 'time': {'type': 'number'}}, 'required': ['principal', 'rate', 'time']}}]}
Finance agent-info: {'name': 'finance-agent', 'description': 'Handles financial calculations like simple interest', 'endpoint': 'http://localhost:8001/invoke'}


In [3]:
# 8) Executable Cell: Invoke baseline tools directly
ok_math = math_mod.invoke({'tool': 'add', 'args': {'a': 7, 'b': 5}})
bad_math = math_mod.invoke({'tool': 'subtract', 'args': {'a': 7, 'b': 5}})
missing_math = math_mod.invoke({'tool': 'add', 'args': {'a': 7}})

ok_finance = finance_mod.invoke({
    'args': {'principal': 1000, 'rate': 5, 'time': 2}
})
missing_finance = finance_mod.invoke({'args': {'principal': 1000, 'rate': 5}})

print('math ok      ->', ok_math)
print('math unknown ->', bad_math)
print('math missing ->', missing_math)
print('finance ok   ->', ok_finance)
print('finance miss ->', missing_finance)

math ok      -> {'status': 'success', 'result': 12}
math unknown -> {'status': 'error', 'message': 'Unknown tool'}
math missing -> {'status': 'error', 'message': "Missing parameter: 'b'"}
finance ok   -> {'status': 'success', 'result': 100.0}
finance miss -> {'status': 'error', 'message': "Missing parameter: 'time'"}


In [4]:
# 9) Executable Cell: Quick static compile check for delayed variants
# This helps validate importability before wiring these files into orchestrators.
for f in ['math_agent_delay.py', 'finance_agent_delay.py', 'math_agent_delay-async.py', 'finance_agent_delay-async.py']:
    p = BASE / f
    src = p.read_text()
    try:
        compile(src, str(p), 'exec')
        print(f'{f:30} compile=OK')
    except SyntaxError as e:
        print(f'{f:30} compile=SYNTAX_ERROR line={e.lineno} msg={e.msg}')

math_agent_delay.py            compile=OK
finance_agent_delay.py         compile=OK
math_agent_delay-async.py      compile=OK
finance_agent_delay-async.py   compile=OK


In [5]:
# 10) Executable Cell: Runtime check for async delayed variant
import asyncio

async_math_mod = load_module('math_agent_delay_async_mod', 'math_agent_delay-async.py')

async def try_async_invoke():
    try:
        result = await async_math_mod.invoke({'tool': 'add', 'args': {'a': 3, 'b': 4}})
        print('async invoke result ->', result)
    except Exception as e:
        print('async invoke raised ->', type(e).__name__, str(e))

await try_async_invoke()

async invoke raised -> TypeError 'coroutine' object is not callable


/home/navinkumar_25_gmail_com/A2A/00_Agents/math_agent_delay-async.py:49: RuntimeWarning: coroutine 'sleep' was never awaited
  await asyncio.sleep(20)(20)


## 6) System Flow Explanation

End-to-end interaction flow in this folder:

1. A controller asks each agent for `/tools` to learn capabilities and input schema.
2. The controller reads `/agent-info` to identify and register reachable invocation endpoints.
3. For user intent, the controller selects an agent/tool pair (e.g., `add` vs `calculate_interest`).
4. The controller sends `POST /invoke` with normalized payload format.
5. Agent executes domain logic and returns success/error envelope.
6. Controller aggregates response and continues orchestration loop.

This is the canonical Agent -> Agent (A2A) interaction skeleton reused in later modules.

## 7) Important Concepts You Might Miss

- The JSON schema under `/tools` is not documentation only; it is machine-usable routing metadata.
- `agent-info.endpoint` is effectively service discovery output and must stay aligned with actual deployment ports.
- Uniform response envelopes (`status/result` vs `status/message`) simplify generic error handling upstream.
- Delay variants model service latency, which is essential for stress-testing orchestration behavior.

## 8) Common Mistakes and Pitfalls

- **Port mismatch risk**: endpoint values in `/agent-info` can drift from actual runtime command ports.
- **Payload shape mismatch**: missing required keys causes `KeyError` paths in invocation handlers.
- **Tool name mismatch**: sending unknown tool names returns explicit error responses.
- **Async runtime fragility**: `math_agent_delay-async.py` contains `await asyncio.sleep(20)(20)`, which compiles but raises an error when executed.
- **Assuming delay == async scalability**: sync `time.sleep` blocks worker threads; async behavior needs event-loop-safe code paths.

## 9) Key Takeaways

- `A2A/00_Agents` defines reusable agent service contracts, not just demo math/finance logic.
- Discovery (`/tools`) + identity (`/agent-info`) + execution (`/invoke`) is the core triad.
- Contract consistency enables later dynamic routing and multi-agent orchestration.
- Latency variants are intentional for resilience and orchestration behavior testing.
- Stabilizing this layer pays off across every subsequent tutorial folder.

## 10) What Is Next (Short Preview)

Next, `01_capability_discovery` introduces a controller notebook that discovers available agents and their capabilities at runtime.
It solves the problem of hardcoded routing by making capability lookup explicit and query-driven.
This matters because orchestration quality depends on discovering the right agent before invocation.
The folder also connects this discovery loop to model-guided decision making in a practical workflow.